## Gold — `dim_area` (perfil das áreas declaradas)

**Origem:** `workspace.silver.cno_areas` → **Destino:** `workspace.gold.dim_area`

- **Modelo:** Star Schema. As colunas categóricas de `cno_areas` (`categoria`, `destinacao`, `tipo_de_obra`, `tipo_de_area`, `tipo_de_area_complementar`) viram uma dimensão de combinações; a metragem fica como **medida** na fato. Como uma obra pode ter várias áreas, o registro da obra se repete na fato para cada área/tipo — a fato referencia esta dimensão via `sk_area`.
- **Grão:** 1 linha por combinação distinta dos 5 atributos (dimensão de combinação, padrão Kimball para atributos que andam juntos).
- **Transformações:**
  - Distinct das combinações — os valores já vêm como **texto oficial** da silver (o CSV do CNO traz descrições, não códigos).
  - Colunas `*_codigo` derivadas por mapeamento reverso dos domínios oficiais RFB (texto → código), preservando também o código do layout.
  - `tipo_de_area_complementar` pode ser nulo (quando `tipo_de_area = Principal`); a fato usa join null-safe (`eqNullSafe`) nesta coluna.
  - `sk_area` = surrogate key sequencial ordenada pelos atributos (determinística).
- **Linhagem:** CSV dados.gov.br → `bronze.cno_areas` → `silver.cno_areas` (textos validados) → `gold.dim_area` (+ códigos derivados).

In [0]:
%run ../shared/_setup

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments, add_table_comment, mapear_valores
from catalogo.cadastro_nacional_obras import (
    DIM_AREA_COMMENTS,
    DIM_AREA_TABLE_COMMENT,
    DOMINIO_CATEGORIA,
    DOMINIO_DESTINACAO,
    DOMINIO_TIPO_DE_OBRA,
    DOMINIO_TIPO_DE_AREA,
    DOMINIO_TIPO_DE_AREA_COMPLEMENTAR,
)

In [0]:
SOURCE_TABLE = "workspace.silver.cno_areas"
TARGET_TABLE = "workspace.gold.dim_area"

# Chaves da combinação (grão da dimensão) — textos oficiais vindos da silver
COLUNAS_COMBINACAO = [
    "categoria",
    "destinacao",
    "tipo_de_obra",
    "tipo_de_area",
    "tipo_de_area_complementar",
]

# Mapeamento reverso dos domínios oficiais: texto -> código do layout RFB
CODIGO_CATEGORIA = {v: k for k, v in DOMINIO_CATEGORIA.items()}
CODIGO_DESTINACAO = {v: k for k, v in DOMINIO_DESTINACAO.items()}
CODIGO_TIPO_DE_OBRA = {v: k for k, v in DOMINIO_TIPO_DE_OBRA.items()}
CODIGO_TIPO_DE_AREA = {v: k for k, v in DOMINIO_TIPO_DE_AREA.items()}
CODIGO_TIPO_DE_AREA_COMPLEMENTAR = {v: k for k, v in DOMINIO_TIPO_DE_AREA_COMPLEMENTAR.items()}

# Contrato de saída gold.dim_area: descrição + código intercalados
COLUNAS_ORDENADAS = [
    "sk_area",
    "categoria",
    "categoria_codigo",
    "destinacao",
    "destinacao_codigo",
    "tipo_de_obra",
    "tipo_de_obra_codigo",
    "tipo_de_area",
    "tipo_de_area_codigo",
    "tipo_de_area_complementar",
    "tipo_de_area_complementar_codigo",
]

In [0]:
df = spark.table(SOURCE_TABLE).select(*COLUNAS_COMBINACAO).distinct()
print(f"Combinações distintas na silver: {df.count():,}")

# Códigos oficiais derivados por mapeamento reverso (texto -> código do layout RFB)
df = (
    df.withColumn("categoria_codigo", mapear_valores("categoria", CODIGO_CATEGORIA))
    .withColumn("destinacao_codigo", mapear_valores("destinacao", CODIGO_DESTINACAO))
    .withColumn("tipo_de_obra_codigo", mapear_valores("tipo_de_obra", CODIGO_TIPO_DE_OBRA))
    .withColumn("tipo_de_area_codigo", mapear_valores("tipo_de_area", CODIGO_TIPO_DE_AREA))
    .withColumn(
        "tipo_de_area_complementar_codigo",
        mapear_valores("tipo_de_area_complementar", CODIGO_TIPO_DE_AREA_COMPLEMENTAR),
    )
)

# Surrogate key determinística: ordenação pelas natural keys da combinação
w = Window.orderBy(*COLUNAS_COMBINACAO)
df = df.withColumn("sk_area", F.row_number().over(w).cast("int"))

df = df.select(*COLUNAS_ORDENADAS)
print(f"Gold: {df.count():,} combinações na dimensão")
display(df.limit(25))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    DIM_AREA_COMMENTS
)
add_table_comment(spark, TARGET_TABLE, DIM_AREA_TABLE_COMMENT)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos_sk = spark.table(TARGET_TABLE).select("sk_area").distinct().count()
distintos_nk = spark.table(TARGET_TABLE).select(*COLUNAS_COMBINACAO).distinct().count()
print(f"Total: {total:,} | SK distintos: {distintos_sk:,} | Combinações distintas: {distintos_nk:,}")
assert total == distintos_sk == distintos_nk, "Quebra de unicidade SK/combinação em dim_area"
display(spark.sql(f"SELECT tipo_de_area, tipo_de_area_codigo, count(*) AS qtd_combinacoes FROM {TARGET_TABLE} GROUP BY tipo_de_area, tipo_de_area_codigo ORDER BY tipo_de_area"))
display(spark.sql(f"SELECT categoria, categoria_codigo, count(*) AS qtd_combinacoes FROM {TARGET_TABLE} GROUP BY categoria, categoria_codigo ORDER BY categoria_codigo"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))